# Original MRA on molab (branch `mra_Og_molab` of the RaCA fork): train and evaluate the baseline

Same run as `kaggle/kaggle_before.ipynb` (original MRA-T, SGN-T warm start, seed 0, batch 1, learning rate 5e-5, 22 epochs, 400 / 150-frame SemanticKITTI subset), moved from a Kaggle T4 to molab's **NVIDIA RTX PRO 6000 Blackwell (96 GB)**.

**Before running**
1. Notebook specs button (app header) → turn the **GPU** on.
2. Fill in the settings cell below: `DATA` (where the 14 GB dataset is) and, recommended, `BACKUP_TO`.
3. Run the cells from top to bottom. Every step skips work that is already done, so re-running after an interruption continues where it stopped (training resumes from the last saved epoch).

**What is different from Kaggle** (details in `molab/README.md`)
- Software: a Blackwell GPU cannot run PyTorch 1.9 / CUDA 11.1, so `molab/install.sh` installs PyTorch 2.7.1 + CUDA 12.8 with matching spconv and torch-scatter builds. MRA's code and the training recipe are unchanged.
- Storage: molab clears the machine when the notebook shuts down (after 12 h, or 90 minutes idle) and does not keep files above 1 GB. Checkpoints live in `/tmp/mra_runs/`; set `BACKUP_TO` or download `results_trained.json` / `latest.pth` before closing.

In [ ]:
# ---- Settings ----
RUN       = "mra_before"                               # which molab/configs/<RUN>.py to train
REPO_URL  = "https://github.com/fafnirLore/RaCA.git"
BRANCH    = "mra_Og_molab"                             # original MRA + kaggle/ + molab/ folders, no RaCA code
EXTRA     = ""                                         # extra train.py args, e.g. "--cfg-options runner.max_epochs=3"

# Where the dataset is. One of:
#   "auto"                               search this machine for .../dataset/sequences (notebook folder, home, /data, /mnt, ...)
#   "/path/to/folder" or an archive      a folder above dataset/ (e.g. .../kitti_subset), or a .zip / .tar(.gz) of it
#   "hf://datasets/<user>/<repo>"        a Hugging Face dataset repository (private: put HF_TOKEN in the notebook's .env)
#   "kaggle://datasets/<user>/<slug>"    a Kaggle dataset            } private: put KAGGLE_USERNAME and KAGGLE_KEY in the
#   "kaggle://kernels/<user>/<slug>"     a Kaggle notebook's output  } notebook's .env (kaggle.com -> Settings -> API -> Create New Token)
DATA      = "auto"

# Where to copy the newest checkpoint, logs and results while training (every 10 minutes), so they survive the session.
#   ""                                   no backup (download the files yourself before the notebook shuts down)
#   "/path/to/folder"                    e.g. a mounted bucket
#   "hf://datasets/<user>/<repo>/runs"   a private Hugging Face dataset repository (HF_TOKEN with write access in .env)
BACKUP_TO = ""

REPO      = "/tmp/RaCA_Og"                             # code
OUT_ROOT  = "/tmp/mra_runs"                            # checkpoints, logs, results: OUT_ROOT/RUN/
print("run:", RUN, "| branch:", BRANCH, "| data:", DATA, "| backup:", BACKUP_TO or "off")

In [ ]:
# Helper: run a shell command and show its output line by line as it is produced.
import os, subprocess

if os.path.exists(".env"):                             # KAGGLE_USERNAME / KAGGLE_KEY / HF_TOKEN, if molab has not loaded them already
    for _line in open(".env"):
        _k, _sep, _v = _line.strip().partition("=")
        if _sep and _k and not _k.startswith("#"):
            os.environ.setdefault(_k.strip(), _v.strip().strip("'\""))

def sh(cmd, **extra_env):
    env = dict(os.environ, RUN=RUN, REPO=REPO, REPO_URL=REPO_URL, BRANCH=BRANCH, EXTRA=EXTRA,
               OUT_ROOT=OUT_ROOT, BACKUP_TO=BACKUP_TO, PYTHONUNBUFFERED="1", **extra_env)
    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {cmd}")
    return "".join(lines)

RUN_DIR = f"{OUT_ROOT}/{RUN}"
MRA_PY  = "/tmp/mra_env/bin/python"

In [ ]:
# 1. Code + software (5-10 min the first time; skipped when already installed in this session)
step1_log = sh("""
set -e
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || { echo "No GPU visible: turn it on with the notebook specs button, then re-run."; exit 1; }
if [ -d "$REPO/.git" ]; then cd "$REPO" && git fetch -q origin "$BRANCH" && git checkout -q "$BRANCH" && git reset -q --hard "origin/$BRANCH"
else git clone -q -b "$BRANCH" "$REPO_URL" "$REPO" && cd "$REPO"; fi
git log --oneline -1    # must show the commit you expect; an existing clone is reset to the branch head on GitHub
bash molab/install.sh
""")

In [ ]:
# 2. Data: find the dataset, rebuild MRA's file layout at /tmp/kitti_full/ (~3 min for the compact subset) and check it
_ = step1_log
step2_log = sh(f'cd "$REPO" && {MRA_PY} molab/prepare_data.py "$DATA" /tmp/kitti_full/', DATA=DATA)
DATA_ROOT = step2_log.strip().splitlines()[-1].split("DATA_ROOT=")[1]

In [ ]:
# 3. Starting weights: ImageNet ResNet-50 + SGN-T, combined into ckpts/warm_start.pth (~1 min).
#    Same starting point as the Kaggle run and the RaCA runs. If training was backed up earlier, bring it back first.
_ = DATA_ROOT
step3_log = sh("""
set -e
cd "$REPO" && bash molab/get_weights.sh
if [ -n "$BACKUP_TO" ]; then /tmp/mra_env/bin/python molab/backup.py pull "$OUT_ROOT/$RUN" "$BACKUP_TO"; fi
""")

In [ ]:
# 4. Smoke test (~1 min): two real training steps and one inference step on this GPU, with the peak memory.
#    If this passes, the software, the data and the starting weights work together.
_ = step3_log
step4_log = sh('cd "$REPO" && source molab/env.sh && PYTHONPATH=$REPO $PY molab/smoke_test.py molab/configs/$RUN.py $SMOKE_EXTRA',
               SMOKE_EXTRA=EXTRA if EXTRA.strip().startswith("--cfg-options") else "")

In [ ]:
# 5. Train. Lines like `Epoch [1][50/400] ... loss: ...` show progress; resumes from the last saved epoch if there is one.
_ = step4_log
import glob
if glob.glob(f"{RUN_DIR}/results_trained.json"):
    step5_log = "already trained and evaluated"
    print(step5_log, "- delete", RUN_DIR, "to train again")
else:
    step5_log = sh('cd "$REPO" && bash molab/train.sh molab/configs/$RUN.py $RUN $EXTRA')

In [ ]:
# 6. Evaluate the trained model on the validation frames (a few minutes)
_ = step5_log
step6_log = sh("""
set -e
cd "$REPO"
[ -s "$OUT_ROOT/$RUN/results_trained.json" ] && echo "results already computed:" && cat "$OUT_ROOT/$RUN/results_trained.json" | head -12 \
  || bash molab/eval.sh molab/configs/$RUN.py "$OUT_ROOT/$RUN/latest.pth" "$OUT_ROOT/$RUN/results_trained.json" "$RUN: trained"
""")

In [ ]:
# 7. Keep the results: final backup (if BACKUP_TO is set) and a copy of the small files next to this notebook.
#    The checkpoint (latest.pth) stays in RUN_DIR: back it up or download it from the file browser before shutdown.
_ = step6_log
import shutil
keep_dir = os.path.join(os.getcwd(), "results", RUN)
os.makedirs(keep_dir, exist_ok=True)
for _f in glob.glob(f"{RUN_DIR}/*.json") + glob.glob(f"{RUN_DIR}/*.log"):
    shutil.copy(_f, keep_dir)
step7_log = sh("""
if [ -n "$BACKUP_TO" ]; then cd "$REPO" && /tmp/mra_env/bin/python molab/backup.py push "$OUT_ROOT/$RUN" "$BACKUP_TO"; fi
du -sh "$OUT_ROOT/$RUN"; ls -la "$OUT_ROOT/$RUN"
""")
print("small result files copied to", keep_dir)